# Day 017 — Exercise 2: Execute Tool

**Goal:** Implement `execute_tool(fn_name, fn_args, registry)` — look up the function in the registry and call it with the arguments the model provided.

In [ ]:
import ollama

In [ ]:
import ast, operator

def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression and return the result as a string."""
    allowed = {
        ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg,
    }
    def _eval(node):
        if isinstance(node, ast.Constant):
            return node.value
        if isinstance(node, ast.BinOp):
            return allowed[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            return allowed[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsafe expression: {ast.dump(node)}")
    result = _eval(ast.parse(expression, mode="eval").body)
    return str(result)

def get_weather(city: str) -> str:
    """Return a simulated current temperature for a city."""
    temperatures = {"london": "12°C", "tokyo": "22°C", "paris": "15°C",
                    "new york": "18°C", "sydney": "24°C"}
    temp = temperatures.get(city.lower(), "20°C")
    return f"The current temperature in {city} is {temp}."

TOOL_REGISTRY = {
    "calculate": calculate,
    "get_weather": get_weather,
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": (
                "Evaluate a mathematical expression and return the numeric result. "
                "Use this for any arithmetic including +, -, *, /, **, and %. "
                "Pass the expression as a string, e.g. '2847 * 193'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A Python math expression, e.g. '12 * 34'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Return the current temperature for a city. "
                "Use this when the user asks about current weather or temperature."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. 'London' or 'Tokyo'",
                    }
                },
                "required": ["city"],
            },
        },
    },
]

## Your Implementation

In [ ]:
def execute_tool(fn_name: str, fn_args: dict, registry: dict) -> str:
    """
    Look up fn_name in registry and call it with **fn_args.

    Returns the tool output as a string.
    Raises KeyError if fn_name is not in the registry.
    """
    # TODO: check fn_name not in registry and raise KeyError
    # TODO: return registry[fn_name](**fn_args)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: execute_tool is defined
    try:
        assert 'execute_tool' in globals()
        passed += 1; print("✅ Check 1: execute_tool is defined")
    except Exception as e:
        print(f"❌ Check 1: not defined — {e}")

    # Check 2: execute_tool calls calculate correctly
    try:
        result = execute_tool("calculate", {"expression": "12 * 34"}, TOOL_REGISTRY)
        assert result == "408", f"expected '408', got {result!r}"
        passed += 1; print("✅ Check 2: calculate('12 * 34') returns '408'")
    except Exception as e:
        print(f"❌ Check 2: calculate — {e}")

    # Check 3: execute_tool calls get_weather correctly
    try:
        result = execute_tool("get_weather", {"city": "Tokyo"}, TOOL_REGISTRY)
        assert "22°C" in result, f"expected '22°C' in result, got {result!r}"
        passed += 1; print("✅ Check 3: get_weather('Tokyo') contains '22°C'")
    except Exception as e:
        print(f"❌ Check 3: get_weather — {e}")

    # Check 4: execute_tool raises KeyError for unknown tool
    try:
        raised = False
        try:
            execute_tool("nonexistent_tool", {}, TOOL_REGISTRY)
        except KeyError:
            raised = True
        assert raised, "execute_tool should raise KeyError for unknown tool name"
        passed += 1; print("✅ Check 4: raises KeyError for unknown tool")
    except Exception as e:
        print(f"❌ Check 4: KeyError — {e}")

    # Check 5: execute_tool evaluates a multi-step expression
    try:
        result = execute_tool("calculate", {"expression": "(100 - 32) * 5/9"}, TOOL_REGISTRY)
        val = float(result)
        assert abs(val - 37.777) < 0.01, f"expected ~37.78 (Celsius), got {result!r}"
        passed += 1; print("✅ Check 5: multi-step expression evaluated correctly")
    except Exception as e:
        print(f"❌ Check 5: multi-step expression — {e}")

    if passed == total:
        print("🎉 Exercise complete!")
    print(f"\nScore: {passed}/{total}")

_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def execute_tool(fn_name: str, fn_args: dict, registry: dict) -> str:
    if fn_name not in registry:
        raise KeyError(f"Unknown tool: {fn_name!r}")
    return registry[fn_name](**fn_args)
```

</details>